# Stage B - Fine-tuning ResNet-18 Classifier
Train an ImageNet-pretrained ResNet-18 expert on 100 classes with freeze/unfreeze scheduling.

In [ ]:
import yaml
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from src.dataset import get_dataloaders
from src.classifier import get_classifier
from src.train_classifier import train_classifier

In [ ]:
with open('../configs/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

device = config['training']['device'] if torch.cuda.is_available() else 'cpu'
model = get_classifier(num_classes=config['dataset']['num_classes'], pretrained=config['classifier']['pretrained']).to(device)
model

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,}')
print(f'Total params: {total:,}')

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    root=config['dataset']['root'],
    image_size=config['dataset']['image_size'],
    batch_size=config['classifier']['batch_size'],
    train_split=config['dataset']['train_split'],
    num_workers=config['training']['num_workers'],
    noisy=False
)

history = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=5,
    learning_rate=config['classifier']['learning_rate'],
    freeze_backbone_epochs=5,
    save_path='../models/resnet_classifier.pth'
)
print('Phase 1 complete: frozen backbone training done.')

In [ ]:
history_ft = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=25,
    learning_rate=config['classifier']['learning_rate'],
    freeze_backbone_epochs=0,
    save_path='../models/resnet_classifier.pth'
)

for k in history:
    history[k].extend(history_ft[k])
print('Phase 2 complete: full fine-tuning done.')

In [ ]:
epochs = range(1, len(history['train_top1']) + 1)
plt.figure(figsize=(10, 4))
plt.plot(epochs, history['train_top1'], label='Train Top-1')
plt.plot(epochs, history['val_top1'], label='Val Top-1')
plt.plot(epochs, history['train_top5'], label='Train Top-5')
plt.plot(epochs, history['val_top5'], label='Val Top-5')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Classifier Accuracy Curves')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        preds = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_targets.append(labels)

all_preds = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()
sample_classes = list(range(10))
mask = [i for i, t in enumerate(all_targets) if t in sample_classes]
cm = confusion_matrix(all_targets[mask], all_preds[mask], labels=sample_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sample_classes)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix (10 sample classes)')
plt.show()

In [ ]:
top5_correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        top5 = logits.topk(5, dim=1).indices.cpu()
        matches = top5.eq(labels.view(-1, 1))
        top5_correct += matches.any(dim=1).sum().item()
        total += labels.size(0)

print(f'Top-5 Accuracy: {100.0 * top5_correct / max(total,1):.2f}%')

## Summary
ResNet-18 has been adapted to 100 classes, trained in frozen and unfrozen phases, and evaluated with top-1/top-5 curves and a class-level confusion matrix.